In [3]:
data = {
  "status": "success",
  "data": {
    "resultType": "matrix",
    "result": [
      {
        "metric": {
          "__name__": "node_cpu_seconds_total",
          "cpu": "0",
          "instance": "web-server-01:9100",
          "job": "node_exporter",
          "mode": "idle",
          "region": "us-east-1",
          "version": "v1.2.0"
        },
        "values": [
          [1741300000, "0.85"],
          [1741300060, "0.82"],
          [1741300120, "0.78"],
          [1741300180, "0.95"],
          [1741300240, "0.98"],
          [1741300300, "0.92"],
          [1741300360, "0.88"],
          [1741300420, "0.85"]
        ]
      }
    ]
  }
}

In [4]:
complex_data_v2 = {
  "status": "success",
  "data": {
    "resultType": "matrix",
    "result": [
      {
        "metric": {
          "__name__": "node_cpu_seconds_total",
          "cpu": "0",
          "instance": "db-prod-02:9100",
          "job": "node_exporter",
          "mode": "idle",
          "region": "us-west-2",
          "env": "production"
        },
        "values": [
          [1741300000, "0.40"],
          [1741300060, "0.45"],
          [1741300120, "0.55"],
          [1741300180, "0.65"],
          [1741300240, "0.75"],
          [1741300300, "0.85"],
          [1741300360, "0.90"],
          [1741300420, "0.95"]
        ]
      }
    ]
  }
}

Importing Libraries

In [5]:
from langgraph.graph import StateGraph, END
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from typing import TypedDict, Annotated, List, Dict, Optional, Literal, IO, Any, Optional
from langgraph.graph import add_messages
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.tools import tool
from urllib.parse import urlparse
import requests
import json
import os

In [6]:
from dotenv import load_dotenv
load_dotenv()

False

In [ ]:
model = ChatOpenAI(
    model="stepfun/step-3.5-flash:free",
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key="",
)

Defining States

In [8]:
class StatMatrices(BaseModel):
    timestamp: int
    cpu_value: float

In [9]:
class MetaData(BaseModel):
    metric_name: str = Field(description="The name of the metric e.g., node_cpu_seconds_total")
    instance: str = Field(description="The server instance ID")
    job: str = Field(description="The job name")
    labels: Dict[str, str] = Field(default_factory=dict)

In [10]:
class MainState(TypedDict):
    jsonFile: str  # The raw Prometheus JSON string
    metaData: MetaData  # Single MetaData object
    statMatrices: List[StatMatrices]  # List of parsed values
    statData: str  # AI ka calculation result
    context: str  # External context like "Sale/Promo event"
    decision: str  # AI ka scaling decision
    report: str  # Final human-readable report

Defining Nodes

In [11]:
def Meta_data_extraction_Node(state: MainState):
    data = json.loads(state['jsonFile'])
    result_item = data['data']['result'][0]
    metric_info = result_item['metric']
    
    # Core fields extract karo
    name = metric_info.pop('__name__', 'unknown')
    inst = metric_info.pop('instance', 'unknown')
    job = metric_info.pop('job', 'unknown')
    
    # Baki bache hue labels ko 'labels' dict mein dalo
    meta = MetaData(
        metric_name=name,
        instance=inst,
        job=job,
        labels=metric_info # Ab region, version, etc. sab yahan aa jayenge
    )
    
    stat_matrices = [
        StatMatrices(timestamp=int(v[0]), cpu_value=float(v[1])) 
        for v in result_item.get('values', [])
    ]
    
    return {"metaData": meta, "statMatrices": stat_matrices}

In [12]:
def statistical_calculations_Node(state: MainState):
    # 1. State se pehle se extract ki gayi list lo
    # Ab json.loads() ki zaroorat nahi hai!
    matrices = state.get('statMatrices', [])
    
    if not matrices:
        return {"statData": "No metric data found."}
    
    # 2. StatMatrices object list se values extract karo
    cpu_values = [m.cpu_value for m in matrices]
    
    # 3. Stats Calculate karo
    avg_cpu = sum(cpu_values) / len(cpu_values)
    max_cpu = max(cpu_values)
    
    # 4. Trend Analysis
    trend = "Increasing" if cpu_values[-1] > cpu_values[0] else "Stable/Decreasing"
    
    # 5. Analysis Summary
    statData = (
        f"Metrics Analysis Report:\n"
        f"- Time Window: {len(cpu_values)} data points\n"
        f"- Average CPU Usage: {avg_cpu:.2f}%\n"
        f"- Peak CPU Usage: {max_cpu:.2f}%\n"
        f"- Trend: {trend}\n"
    )
    
    return {"statData": statData}

In [13]:
def major_decisions_Node(state: MainState):
    print("--- Node Triggered: Major Decisions ---")
    
    # 1. Input preparation
    meta = state.get("metaData")
    stats = state.get("statData")
    context = state.get("context")
    
    # 2. Advanced SRE Persona Prompt
    prompt = f"""
    You are an expert SRE (Site Reliability Engineer) monitoring infrastructure.
    
    SERVER CONTEXT: {meta}
    METRIC ANALYSIS: {stats}
    BUSINESS CONTEXT: {context}
    
    Based on the analysis, decide: 
    1. Do we need to scale? (Yes/No)
    2. What action should be taken?
    3. Justify your decision in one sentence.
    
    FORMAT: Respond only with the decision clearly.
    """
    
    # 3. Invoke LLM
    decision = model.invoke(prompt).content
    return {"decision": decision}

Creating Graph

In [14]:
graph = StateGraph(MainState)

#adding Nodes
graph.add_node("meta_data_extraction", Meta_data_extraction_Node)
graph.add_node("statistical_calculations", statistical_calculations_Node)
# graph.add_node("external_context", external_context_Node)
graph.add_node("major_decisions", major_decisions_Node)
# graph.add_node("report", report_Node)

# add edges

graph.set_entry_point("meta_data_extraction")
graph.add_edge("meta_data_extraction", "statistical_calculations")
# graph.add_edge("meta_data_extraction", "external_context")

graph.add_edge("statistical_calculations", "major_decisions")
# graph.add_edge("external_context", "major_decisions")

# graph.add_edge("major_decisions", "report")
graph.set_finish_point("major_decisions")

app = graph.compile()

In [15]:
app.get_graph().print_ascii()

        +-----------+        
        | __start__ |        
        +-----------+        
              *              
              *              
              *              
  +----------------------+   
  | meta_data_extraction |   
  +----------------------+   
              *              
              *              
              *              
+--------------------------+ 
| statistical_calculations | 
+--------------------------+ 
              *              
              *              
              *              
    +-----------------+      
    | major_decisions |      
    +-----------------+      
              *              
              *              
              *              
        +---------+          
        | __end__ |          
        +---------+          


Testing

In [16]:
# 1. JSON ko string mein convert karo taaki state mein ja sake
import json
json_string = json.dumps(data)

# 2. Graph Invoke (Testing the Entry Point)
# Hum "meta_data_extraction" se shuru karenge
state = {"jsonFile": json_string, "context": "Normal operation."}


print("--- Starting Pipeline ---")
for event in app.stream(state):
    for node_name, output in event.items():
        print(f"Node '{node_name}' finished. Output: {output}")
# Graph execute karo
result = app.invoke(state)

# 3. Output Check karo
print("--- Final Result ---")
print("metaData:", result.get("metaData"))
print("statMatrices:", result.get("statMatrices"))
print("statData:", result.get("statData"))
print("context:", result.get("context"))
print("major_decisions:", result.get("decision"))

--- Starting Pipeline ---
Node 'meta_data_extraction' finished. Output: {'metaData': MetaData(metric_name='node_cpu_seconds_total', instance='web-server-01:9100', job='node_exporter', labels={'cpu': '0', 'mode': 'idle', 'region': 'us-east-1', 'version': 'v1.2.0'}), 'statMatrices': [StatMatrices(timestamp=1741300000, cpu_value=0.85), StatMatrices(timestamp=1741300060, cpu_value=0.82), StatMatrices(timestamp=1741300120, cpu_value=0.78), StatMatrices(timestamp=1741300180, cpu_value=0.95), StatMatrices(timestamp=1741300240, cpu_value=0.98), StatMatrices(timestamp=1741300300, cpu_value=0.92), StatMatrices(timestamp=1741300360, cpu_value=0.88), StatMatrices(timestamp=1741300420, cpu_value=0.85)]}
Node 'statistical_calculations' finished. Output: {'statData': 'Metrics Analysis Report:\n- Time Window: 8 data points\n- Average CPU Usage: 0.88%\n- Peak CPU Usage: 0.98%\n- Trend: Stable/Decreasing\n'}
--- Node Triggered: Major Decisions ---
Node 'major_decisions' finished. Output: {'decision': 'N

In [17]:
print("major_decisions:", result.get("decision"))

major_decisions: 1. No  
2. Continue monitoring  
3. CPU usage is very low (0.88% average) and stable/decreasing, indicating no resource pressure requiring scaling.
